In [ ]:
from pathlib import Path
import os

root = Path("/kaggle/input")

for item in root.iterdir():
    print(item)

In [ ]:
from pathlib import Path

root = Path("/kaggle/input/competitions")

for item in root.iterdir():
    print(item)
    

In [ ]:
from pathlib import Path

root = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

for item in sorted(root.iterdir()):
    print(item)
    

In [ ]:
import pandas as pd
from pathlib import Path

root = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

for filename in [
    "train.csv",
    "test.csv",
    "train_series.csv",
    "test_series.csv",
    "sample_submission.csv"
]:
    print("\n" + "=" * 70)
    print(filename)

    df = pd.read_csv(root / filename)

    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())

    print("\nFirst 3 rows:")
    display(df.head(3))
    

In [ ]:
import pandas as pd
from pathlib import Path

root = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

train = pd.read_csv(root / "train.csv")

print("Shape:", train.shape)
print("\nColumns:")
print(train.columns.tolist())

print("\nFirst 5 rows:")
display(train.head())

In [ ]:
import pandas as pd
from pathlib import Path

root = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
train = pd.read_csv(root / "train.csv")

targets = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

print("Shape:", train.shape)

print("\nData types:")
print(train[targets].dtypes)

print("\nMissing values:")
print(train[targets].isna().sum())

print("\nUnique values for every target:")
for col in targets:
    print(f"\n--- {col} ---")
    print(train[col].value_counts(dropna=False).sort_index())

In [ ]:
import pandas as pd
from pathlib import Path

root = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

train = pd.read_csv(root / "train.csv")
test = pd.read_csv(root / "test.csv")
train_series = pd.read_csv(root / "train_series.csv")
test_series = pd.read_csv(root / "test_series.csv")

print("TRAIN")
print(train.shape)

print("\nTEST")
print(test.shape)

print("\nTRAIN SERIES")
print(train_series.shape)

print("\nTEST SERIES")
print(test_series.shape)

print("\nLabeled train rows:")
print(train["ACL"].notna().sum())

print("\nFirst 5 labeled StudyInstanceUIDs:")
print(train.loc[train["ACL"].notna(), "StudyInstanceUID"].head().tolist())

print("\nFirst 5 unlabeled StudyInstanceUIDs:")
print(train.loc[train["ACL"].isna(), "StudyInstanceUID"].head().tolist())

print("\nLast 5 rows of train:")
display(train.tail())

In [ ]:
import pandas as pd
from pathlib import Path

root = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

train_series = pd.read_csv(root / "train_series.csv")

print("Shape:", train_series.shape)

print("\nColumns:")
print(train_series.columns.tolist())

print("\nFirst 10 rows:")
display(train_series.head(10))

print("\nData types:")
print(train_series.dtypes)

In [ ]:
print("\nNumber of unique studies:")
print(train_series["StudyInstanceUID"].nunique())

print("\nNumber of unique series:")
print(train_series["SeriesInstanceUID"].nunique())

print("\nSeries per study:")
series_per_study = train_series.groupby("StudyInstanceUID")["SeriesInstanceUID"].nunique()

print(series_per_study.describe())

print("\nFirst 20 studies:")
display(series_per_study.head(20))

In [ ]:
from pathlib import Path
import pandas as pd

root = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

train_series = pd.read_csv(root / "train_series.csv")

print("Shape:", train_series.shape)

print("\nColumns:")
print(train_series.columns.tolist())

print("\nFirst 20 rows:")
display(train_series.head(20))

In [ ]:
import os
from pathlib import Path
import pandas as pd

root = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

study_id = pd.read_csv(root / "train_series.csv").iloc[0]["StudyInstanceUID"]
series_id = pd.read_csv(root / "train_series.csv").iloc[0]["SeriesInstanceUID"]

print("StudyInstanceUID:")
print(study_id)

print("\nSeriesInstanceUID:")
print(series_id)

print("\nSearching for DICOM files...")

series_path = root / "train_series" / str(study_id) / str(series_id)

print("Expected series path:")
print(series_path)

if series_path.exists():
    files = sorted([p for p in series_path.rglob("*") if p.is_file()])
    print("\nNumber of files:", len(files))

    print("\nFirst 10 files:")
    for f in files[:10]:
        print(f)
else:
    print("\nSeries path does not exist at this exact structure.")
    print("We will inspect the directory structure next.")

In [ ]:
import pydicom
import matplotlib.pyplot as plt
from pathlib import Path

root = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

study_id = "1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260"
series_id = "1.2.826.0.1.3680043.8.498.12343110195036213483454091715412333772"

series_path = root / "train_series" / study_id / series_id

files = sorted(series_path.glob("*.dcm"))

print("Number of DICOM files:", len(files))

# Load all slices
slices = []

for f in files:
    ds = pydicom.dcmread(f)
    slices.append((ds, ds.pixel_array))

# Show metadata from first slice
ds = slices[0][0]

print("\nDICOM metadata")
print("Rows:", ds.Rows)
print("Columns:", ds.Columns)
print("Modality:", getattr(ds, "Modality", "N/A"))
print("Series Description:", getattr(ds, "SeriesDescription", "N/A"))
print("Instance Number:", getattr(ds, "InstanceNumber", "N/A"))
print("Slice Thickness:", getattr(ds, "SliceThickness", "N/A"))
print("Pixel Spacing:", getattr(ds, "PixelSpacing", "N/A"))

# Show 6 evenly spaced slices
indices = [0, 4, 8, 12, 16, 21]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, idx in zip(axes.ravel(), indices):
    image = slices[idx][1]
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Slice {idx + 1}/{len(slices)}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import pydicom
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

root = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

study_id = "1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260"
series_id = "1.2.826.0.1.3680043.8.498.12343110195036213483454091715412333772"

series_path = root / "train_series" / study_id / series_id

records = []

for f in series_path.glob("*.dcm"):
    ds = pydicom.dcmread(f)

    position = getattr(ds, "ImagePositionPatient", None)
    orientation = getattr(ds, "ImageOrientationPatient", None)
    instance = getattr(ds, "InstanceNumber", None)

    records.append({
        "file": f,
        "instance": instance,
        "position": position,
        "orientation": orientation,
    })

print("Number of slices:", len(records))

print("\nFirst 10 metadata records:")
for r in records[:10]:
    print(
        "Instance:", r["instance"],
        "| Position:", r["position"],
        "| File:", r["file"].name
    )

# Sort using ImagePositionPatient when available.
if all(r["position"] is not None for r in records):
    records = sorted(
        records,
        key=lambda r: tuple(float(x) for x in r["position"])
    )
    sorting_method = "ImagePositionPatient"
else:
    records = sorted(
        records,
        key=lambda r: (
            r["instance"] if r["instance"] is not None else 0
        )
    )
    sorting_method = "InstanceNumber"

print("\nSorting method:", sorting_method)

print("\nOrdered slices:")
for i, r in enumerate(records):
    print(
        i,
        "| Instance:", r["instance"],
        "| Position:", r["position"]
    )

# Visualize ordered slices
images = []

for r in records:
    ds = pydicom.dcmread(r["file"])
    images.append(ds.pixel_array.astype(np.float32))

indices = np.linspace(
    0,
    len(images) - 1,
    min(6, len(images)),
    dtype=int
)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, idx in zip(axes.ravel(), indices):
    ax.imshow(images[idx], cmap="gray")
    ax.set_title(f"Ordered slice {idx + 1}/{len(images)}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import pydicom
import numpy as np
from pathlib import Path


def load_dicom_series(series_path):
    """
    Load a DICOM MRI series and order slices using
    ImageOrientationPatient + ImagePositionPatient.
    """

    series_path = Path(series_path)

    dicom_files = list(series_path.glob("*.dcm"))

    if not dicom_files:
        raise FileNotFoundError(f"No DICOM files found in: {series_path}")

    records = []

    for file_path in dicom_files:
        ds = pydicom.dcmread(file_path)

        if not hasattr(ds, "ImagePositionPatient"):
            raise ValueError(
                f"ImagePositionPatient missing: {file_path.name}"
            )

        if not hasattr(ds, "ImageOrientationPatient"):
            raise ValueError(
                f"ImageOrientationPatient missing: {file_path.name}"
            )

        position = np.asarray(
            ds.ImagePositionPatient,
            dtype=np.float64
        )

        orientation = np.asarray(
            ds.ImageOrientationPatient,
            dtype=np.float64
        )

        row_direction = orientation[:3]
        column_direction = orientation[3:]

        slice_normal = np.cross(
            row_direction,
            column_direction
        )

        slice_position = np.dot(
            position,
            slice_normal
        )

        records.append({
            "file": file_path,
            "dataset": ds,
            "position": position,
            "slice_position": slice_position,
            "instance": getattr(ds, "InstanceNumber", None),
        })

    # Sort by physical slice position
    records.sort(key=lambda x: x["slice_position"])

    images = []

    for record in records:
        image = record["dataset"].pixel_array.astype(np.float32)
        images.append(image)

    volume = np.stack(images, axis=0)

    metadata = {
        "num_slices": len(records),
        "height": volume.shape[1],
        "width": volume.shape[2],
        "modality": getattr(records[0]["dataset"], "Modality", None),
        "series_description": getattr(
            records[0]["dataset"],
            "SeriesDescription",
            None
        ),
        "pixel_spacing": getattr(
            records[0]["dataset"],
            "PixelSpacing",
            None
        ),
        "slice_thickness": getattr(
            records[0]["dataset"],
            "SliceThickness",
            None
        ),
        "slice_positions": [
            r["slice_position"] for r in records
        ],
        "instance_numbers": [
            r["instance"] for r in records
        ],
    }

    return volume, metadata

In [ ]:
root = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection"
)

study_id = (
    "1.2.826.0.1.3680043.8.498."
    "10004873229099053869093324292195817260"
)

series_id = (
    "1.2.826.0.1.3680043.8.498."
    "12343110195036213483454091715412333772"
)

series_path = (
    root
    / "train_series"
    / study_id
    / series_id
)

volume, metadata = load_dicom_series(series_path)

print("Volume shape:", volume.shape)

print("\nMetadata:")
for key, value in metadata.items():
    if key not in ["slice_positions"]:
        print(f"{key}: {value}")

print("\nFirst 5 physical positions:")
print(metadata["slice_positions"][:5])

print("\nLast 5 physical positions:")
print(metadata["slice_positions"][-5:])

In [ ]:
import pandas as pd
from pathlib import Path

root = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection"
)

train_series = pd.read_csv(root / "train_series.csv")
train = pd.read_csv(root / "train.csv")

print("Total series rows:", len(train_series))
print("Unique studies:", train_series["StudyInstanceUID"].nunique())
print("Unique series:", train_series["SeriesInstanceUID"].nunique())

print("\n" + "="*60)
print("ANATOMICAL PLANE")
print("="*60)
display(
    train_series["Anatomical_Plane"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\n" + "="*60)
print("FLUID SENSITIVE")
print("="*60)
display(
    train_series["Fluid_Sensitive"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\n" + "="*60)
print("FAT SUPPRESSION")
print("="*60)
display(
    train_series["Fat_Suppression"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\n" + "="*60)
print("PLANE × FLUID SENSITIVE × FAT SUPPRESSION")
print("="*60)

display(
    train_series
    .groupby(
        ["Anatomical_Plane",
         "Fluid_Sensitive",
         "Fat_Suppression"],
        dropna=False
    )
    .size()
    .reset_index(name="number_of_series")
    .sort_values("number_of_series", ascending=False)
)

In [ ]:
# How many series of each type does a typical study contain?

study_series_summary = (
    train_series
    .groupby(["StudyInstanceUID", "Anatomical_Plane"])
    .size()
    .unstack(fill_value=0)
)

print("Series counts per study by anatomical plane:")
display(study_series_summary.describe())

print("\nFirst 20 studies:")
display(study_series_summary.head(20))

In [ ]:
# Create one row per study showing which series types are present

series_types = (
    train_series
    .groupby(
        [
            "StudyInstanceUID",
            "Anatomical_Plane",
            "Fluid_Sensitive",
            "Fat_Suppression"
        ]
    )
    .size()
    .reset_index(name="count")
)

# Turn the six sequence combinations into columns
study_matrix = (
    series_types
    .assign(
        sequence_type=lambda x:
        x["Anatomical_Plane"].astype(str)
        + "_F"
        + x["Fluid_Sensitive"].astype(str)
        + "_FS"
        + x["Fat_Suppression"].astype(str)
    )
    .pivot_table(
        index="StudyInstanceUID",
        columns="sequence_type",
        values="count",
        fill_value=0
    )
)

print("Number of studies:", len(study_matrix))

print("\nHow many sequence types does each study contain?")

num_sequence_types = (study_matrix > 0).sum(axis=1)

display(
    num_sequence_types
    .value_counts()
    .sort_index()
    .to_frame("number_of_studies")
)

print("\nSequence presence across studies:")

presence = (study_matrix > 0).mean().sort_values(ascending=False)

display(
    (presence * 100)
    .round(2)
    .to_frame("percentage_of_studies")
)

print("\nFirst 20 studies:")
display(study_matrix.head(20))

In [ ]:
# Exact study-level sequence completeness

num_sequence_types = (study_matrix > 0).sum(axis=1)

print("=" * 60)
print("NUMBER OF DIFFERENT SEQUENCE TYPES PER STUDY")
print("=" * 60)

display(
    num_sequence_types
    .value_counts()
    .sort_index()
    .rename("number_of_studies")
    .to_frame()
)

print("\n" + "=" * 60)
print("PERCENTAGE OF STUDIES")
print("=" * 60)

display(
    (
        num_sequence_types
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
        .rename("percentage")
        .to_frame()
    )
)

print("\n" + "=" * 60)
print("TOTAL NUMBER OF STUDIES")
print("=" * 60)

print(len(study_matrix))

In [ ]:
print("=" * 60)
print("PRESENCE OF EACH SEQUENCE TYPE")
print("=" * 60)

presence = (study_matrix > 0).sum(axis=0)
total_studies = len(study_matrix)

summary = pd.DataFrame({
    "studies_with_sequence": presence,
    "percentage": (presence / total_studies * 100).round(2)
})

display(summary.sort_values("percentage", ascending=False))

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

root = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection"
)

train_series = pd.read_csv(root / "train_series.csv")

# Only the four common sequence types for now
common_types = [
    ("Axial", 1, 1),
    ("Sagittal", 0, 0),
    ("Coronal", 1, 1),
    ("Sagittal", 1, 1),
]

rows = []

for plane, fluid, fat in common_types:
    
    subset = train_series[
        (train_series["Anatomical_Plane"] == plane) &
        (train_series["Fluid_Sensitive"] == fluid) &
        (train_series["Fat_Suppression"] == fat)
    ]

    # Count actual DICOM files for up to 100 series first
    for _, r in subset.head(100).iterrows():

        series_path = (
            root
            / "train_series"
            / str(r["StudyInstanceUID"])
            / str(r["SeriesInstanceUID"])
        )

        n_files = len(list(series_path.glob("*.dcm")))

        rows.append({
            "sequence_type": f"{plane}_F{fluid}_FS{fat}",
            "StudyInstanceUID": r["StudyInstanceUID"],
            "SeriesInstanceUID": r["SeriesInstanceUID"],
            "num_slices": n_files
        })

slice_summary = pd.DataFrame(rows)

print("Sampled series:", len(slice_summary))

display(
    slice_summary
    .groupby("sequence_type")["num_slices"]
    .describe()
)

In [ ]:
import pandas as pd
from pathlib import Path

root = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection"
)

train = pd.read_csv(root / "train.csv")
train_series = pd.read_csv(root / "train_series.csv")

targets = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

# These are the studies with known labels
labelled = train[train[targets].notna().all(axis=1)].copy()

print("Total studies:", len(train))
print("Fully labelled studies:", len(labelled))

# Sequence availability among ONLY labelled studies
common_types = [
    ("Axial", 1, 1),
    ("Sagittal", 0, 0),
    ("Coronal", 1, 1),
    ("Sagittal", 1, 1),
]

labelled_series = train_series[
    train_series["StudyInstanceUID"].isin(labelled["StudyInstanceUID"])
].copy()

print("\nLabelled-study sequence availability:")

for plane, fluid, fat in common_types:
    mask = (
        (labelled_series["Anatomical_Plane"] == plane) &
        (labelled_series["Fluid_Sensitive"] == fluid) &
        (labelled_series["Fat_Suppression"] == fat)
    )

    n = labelled_series.loc[mask, "StudyInstanceUID"].nunique()
    pct = n / len(labelled) * 100

    print(
        f"{plane}_F{fluid}_FS{fat}: "
        f"{n}/{len(labelled)} studies ({pct:.2f}%)"
    )

print("\nLabelled studies:")
display(labelled[["StudyInstanceUID"] + targets].head())

In [ ]:
import pandas as pd
from pathlib import Path

root = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

train = pd.read_csv(root / "train.csv")
train_series = pd.read_csv(root / "train_series.csv")

targets = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

# Exactly the studies with complete labels
labelled = train[train[targets].notna().all(axis=1)].copy()

print("Total studies:", len(train))
print("Fully labelled studies:", len(labelled))

labelled_series = train_series[
    train_series["StudyInstanceUID"].isin(labelled["StudyInstanceUID"])
].copy()

sequence_types = [
    ("Axial", 1, 1),
    ("Sagittal", 0, 0),
    ("Coronal", 1, 1),
    ("Sagittal", 1, 1),
    ("Coronal", 0, 0),
    ("Axial", 0, 0),
]

print("\nSequence availability among the 58 labelled studies:\n")

for plane, fluid, fat in sequence_types:
    mask = (
        (labelled_series["Anatomical_Plane"] == plane) &
        (labelled_series["Fluid_Sensitive"] == fluid) &
        (labelled_series["Fat_Suppression"] == fat)
    )

    n_studies = labelled_series.loc[
        mask, "StudyInstanceUID"
    ].nunique()

    percentage = 100 * n_studies / len(labelled)

    print(
        f"{plane}_F{fluid}_FS{fat}: "
        f"{n_studies}/58 ({percentage:.2f}%)"
    )

print("\nLabel distribution in the 58 labelled studies:")

distribution = pd.DataFrame({
    "positive": labelled[targets].sum(),
    "negative": (1 - labelled[targets]).sum()
})

distribution["positive_rate_%"] = (
    distribution["positive"] / len(labelled) * 100
).round(2)

display(distribution)

In [ ]:
print("\nSequence availability among the 58 labelled studies:\n")

for plane, fluid, fat in [
    ("Axial", 1, 1),
    ("Sagittal", 0, 0),
    ("Coronal", 1, 1),
    ("Sagittal", 1, 1),
    ("Coronal", 0, 0),
    ("Axial", 0, 0),
]:
    mask = (
        (labelled_series["Anatomical_Plane"] == plane) &
        (labelled_series["Fluid_Sensitive"] == fluid) &
        (labelled_series["Fat_Suppression"] == fat)
    )

    n = labelled_series.loc[
        mask, "StudyInstanceUID"
    ].nunique()

    print(
        f"{plane}_F{fluid}_FS{fat}: "
        f"{n}/58 ({100*n/58:.2f}%)"
    )

In [ ]:
sequence_cols = [
    ("Axial", 1, 1, "Axial_F1_FS1"),
    ("Sagittal", 0, 0, "Sagittal_F0_FS0"),
    ("Coronal", 1, 1, "Coronal_F1_FS1"),
    ("Sagittal", 1, 1, "Sagittal_F1_FS1"),
]

print("Duplicate series counts within labelled studies:\n")

for plane, fluid, fat, name in sequence_cols:

    subset = labelled_series[
        (labelled_series["Anatomical_Plane"] == plane) &
        (labelled_series["Fluid_Sensitive"] == fluid) &
        (labelled_series["Fat_Suppression"] == fat)
    ]

    counts = (
        subset.groupby("StudyInstanceUID")
        .size()
    )

    print(f"\n{name}")
    print("Studies with sequence:", counts.size)
    print("Studies with >1 series:", int((counts > 1).sum()))
    print("Maximum series in one study:", int(counts.max()))
    print("\nDistribution:")
    print(counts.value_counts().sort_index())

In [ ]:
import pandas as pd
from pathlib import Path

root = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection"
)

train = pd.read_csv(root / "train.csv")
train_series = pd.read_csv(root / "train_series.csv")

targets = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

# ---------------------------------------------------------
# 1. Keep only completely labelled studies
# ---------------------------------------------------------

labelled = train[
    train[targets].notna().all(axis=1)
].copy()

print("Labelled studies:", len(labelled))

# ---------------------------------------------------------
# 2. Define the four sequence types for Baseline V1
# ---------------------------------------------------------

sequence_types = [
    ("Axial", 1, 1, "axial_f1_fs1"),
    ("Sagittal", 0, 0, "sagittal_f0_fs0"),
    ("Coronal", 1, 1, "coronal_f1_fs1"),
    ("Sagittal", 1, 1, "sagittal_f1_fs1"),
]

# ---------------------------------------------------------
# 3. Find candidate series for every study/sequence type
# ---------------------------------------------------------

rows = []

for plane, fluid, fat, name in sequence_types:

    subset = train_series[
        (train_series["Anatomical_Plane"] == plane) &
        (train_series["Fluid_Sensitive"] == fluid) &
        (train_series["Fat_Suppression"] == fat)
    ].copy()

    for study_id, group in subset.groupby("StudyInstanceUID"):

        # Only studies in our labelled set
        if study_id not in set(labelled["StudyInstanceUID"]):
            continue

        candidates = []

        for _, series_row in group.iterrows():

            series_path = (
                root
                / "train_series"
                / str(study_id)
                / str(series_row["SeriesInstanceUID"])
            )

            n_slices = len(
                list(series_path.glob("*.dcm"))
            )

            candidates.append({
                "SeriesInstanceUID":
                    series_row["SeriesInstanceUID"],
                "num_slices":
                    n_slices
            })

        # Pick longest series
        candidates = sorted(
            candidates,
            key=lambda x: x["num_slices"],
            reverse=True
        )

        if candidates:
            selected = candidates[0]

            rows.append({
                "StudyInstanceUID": study_id,
                "sequence_type": name,
                "SeriesInstanceUID":
                    selected["SeriesInstanceUID"],
                "num_slices":
                    selected["num_slices"],
                "num_candidate_series":
                    len(candidates)
            })

selected_series = pd.DataFrame(rows)

print("\nSelected series:")
display(selected_series.head(20))

print("\nRows:", len(selected_series))

print("\nSelected series by sequence type:")
display(
    selected_series["sequence_type"]
    .value_counts()
)

In [ ]:
# ---------------------------------------------------------
# 4. Pivot into one row per study
# ---------------------------------------------------------

study_metadata = labelled[
    ["StudyInstanceUID", "Report"] + targets
].copy()

series_pivot = selected_series.pivot(
    index="StudyInstanceUID",
    columns="sequence_type",
    values="SeriesInstanceUID"
).reset_index()

study_metadata = study_metadata.merge(
    series_pivot,
    on="StudyInstanceUID",
    how="left"
)

print("Final metadata table shape:", study_metadata.shape)

display(study_metadata.head())

print("\nMissing selected series:")
display(
    study_metadata[
        [
            "axial_f1_fs1",
            "sagittal_f0_fs0",
            "coronal_f1_fs1",
            "sagittal_f1_fs1"
        ]
    ].isna().sum()
)

In [ ]:
output_path = Path("/kaggle/working/study_metadata_v1.csv")

study_metadata.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

In [ ]:
import pandas as pd
from pathlib import Path

root = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection"
)

metadata_path = Path("/kaggle/working/study_metadata_v1.csv")

study_metadata = pd.read_csv(metadata_path)

print("Shape:", study_metadata.shape)
print("\nColumns:")
print(study_metadata.columns.tolist())

print("\nMissing values:")
display(
    study_metadata.isna().sum().to_frame("missing")
)

print("\nSelected series columns:")
display(
    study_metadata[
        [
            "StudyInstanceUID",
            "axial_f1_fs1",
            "sagittal_f0_fs0",
            "coronal_f1_fs1",
            "sagittal_f1_fs1"
        ]
    ].head(10)
)

In [ ]:
from pathlib import Path
import pandas as pd
import pydicom

root = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection"
)

study_metadata = pd.read_csv(
    "/kaggle/working/study_metadata_v1.csv"
)

sequence_columns = [
    "axial_f1_fs1",
    "sagittal_f0_fs0",
    "coronal_f1_fs1",
    "sagittal_f1_fs1",
]

results = []

for _, row in study_metadata.iterrows():

    study_id = row["StudyInstanceUID"]

    for seq in sequence_columns:

        series_id = row[seq]

        if pd.isna(series_id):
            results.append({
                "StudyInstanceUID": study_id,
                "sequence": seq,
                "exists": False,
                "num_dicom": 0,
                "readable": False,
            })
            continue

        series_path = (
            root
            / "train_series"
            / str(study_id)
            / str(series_id)
        )

        dcm_files = list(series_path.glob("*.dcm"))

        readable = False

        if dcm_files:
            try:
                ds = pydicom.dcmread(
                    dcm_files[0],
                    stop_before_pixels=True
                )
                readable = True
            except Exception:
                readable = False

        results.append({
            "StudyInstanceUID": study_id,
            "sequence": seq,
            "exists": series_path.exists(),
            "num_dicom": len(dcm_files),
            "readable": readable,
        })

integrity = pd.DataFrame(results)

print("Total checks:", len(integrity))

print("\nDirectory existence:")
display(
    integrity["exists"].value_counts(dropna=False)
)

print("\nReadable DICOM:")
display(
    integrity["readable"].value_counts(dropna=False)
)

print("\nDICOM slice-count summary:")
display(
    integrity
    .groupby("sequence")["num_dicom"]
    .describe()
)

print("\nAny problematic entries:")
display(
    integrity[
        (~integrity["exists"]) |
        (~integrity["readable"]) |
        (integrity["num_dicom"] == 0)
    ]
)

In [ ]:
# Pick the first problematic entry automatically

bad = integrity[
    (~integrity["exists"]) |
    (~integrity["readable"]) |
    (integrity["num_dicom"] == 0)
].iloc[0]

print("Problematic study:")
print("StudyInstanceUID:", bad["StudyInstanceUID"])
print("Sequence:", bad["sequence"])

study_id = bad["StudyInstanceUID"]

series_id = study_metadata.loc[
    study_metadata["StudyInstanceUID"] == study_id,
    bad["sequence"]
].iloc[0]

print("\nSelected SeriesInstanceUID:")
print(series_id)

study_path = root / "train_series" / str(study_id)

print("\nStudy directory exists:", study_path.exists())
print("Study directory:", study_path)

if study_path.exists():
    print("\nActual contents:")
    for item in sorted(study_path.iterdir()):
        print(item)

In [ ]:
study_id = "1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645"

row = study_metadata[
    study_metadata["StudyInstanceUID"] == study_id
].iloc[0]

sequence_columns = [
    "axial_f1_fs1",
    "sagittal_f0_fs0",
    "coronal_f1_fs1",
    "sagittal_f1_fs1",
]

print("Selected series for this study:\n")

for col in sequence_columns:
    print(f"{col}: {row[col]}")

In [ ]:
print("\nActual series from train_series.csv:\n")

actual = train_series[
    train_series["StudyInstanceUID"] == study_id
]

display(
    actual[
        [
            "SeriesInstanceUID",
            "Fluid_Sensitive",
            "Fat_Suppression",
            "Anatomical_Plane"
        ]
    ]
)

In [ ]:
import pandas as pd
from pathlib import Path

metadata_path = Path("/kaggle/working/study_metadata_v1.csv")

study_metadata = pd.read_csv(metadata_path)

sequence_columns = [
    "axial_f1_fs1",
    "sagittal_f0_fs0",
    "coronal_f1_fs1",
    "sagittal_f1_fs1",
]

for col in sequence_columns:
    study_metadata[col + "_available"] = (
        study_metadata[col].notna().astype(int)
    )

availability_columns = [
    col + "_available"
    for col in sequence_columns
]

print("Sequence availability:")
display(
    study_metadata[availability_columns]
    .value_counts()
    .reset_index(name="number_of_studies")
)

print("\nMissing sequence count:")
for col in sequence_columns:
    print(
        f"{col}: "
        f"{study_metadata[col].isna().sum()} missing"
    )

new_path = Path(
    "/kaggle/working/study_metadata_v2.csv"
)

study_metadata.to_csv(
    new_path,
    index=False
)

print("\nSaved:", new_path)

In [ ]:
import numpy as np
import pandas as pd
import pydicom
import torch

from pathlib import Path
from torch.utils.data import Dataset
import torch.nn.functional as F


ROOT = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection"
)

METADATA_PATH = Path(
    "/kaggle/working/study_metadata_v2.csv"
)

metadata = pd.read_csv(METADATA_PATH)


TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]


SEQUENCES = [
    "axial_f1_fs1",
    "sagittal_f0_fs0",
    "coronal_f1_fs1",
    "sagittal_f1_fs1",
]


def load_dicom_volume(series_path):
    """
    Load one DICOM series.

    Slice ordering:
        ImageOrientationPatient
        +
        ImagePositionPatient

    Returns:
        volume: numpy array [N, H, W]
    """

    series_path = Path(series_path)

    files = list(series_path.glob("*.dcm"))

    if not files:
        raise FileNotFoundError(
            f"No DICOM files found: {series_path}"
        )

    records = []

    for file_path in files:

        ds = pydicom.dcmread(file_path)

        if not hasattr(ds, "ImagePositionPatient"):
            raise ValueError(
                f"Missing ImagePositionPatient: {file_path}"
            )

        if not hasattr(ds, "ImageOrientationPatient"):
            raise ValueError(
                f"Missing ImageOrientationPatient: {file_path}"
            )

        position = np.asarray(
            ds.ImagePositionPatient,
            dtype=np.float64
        )

        orientation = np.asarray(
            ds.ImageOrientationPatient,
            dtype=np.float64
        )

        row_direction = orientation[:3]
        column_direction = orientation[3:]

        normal = np.cross(
            row_direction,
            column_direction
        )

        physical_position = np.dot(
            position,
            normal
        )

        records.append(
            (
                physical_position,
                file_path
            )
        )

    records.sort(key=lambda x: x[0])

    images = []

    for _, file_path in records:

        ds = pydicom.dcmread(file_path)

        image = ds.pixel_array.astype(
            np.float32
        )

        images.append(image)

    volume = np.stack(
        images,
        axis=0
    )

    return volume


def sample_slices(volume, num_slices=16):
    """
    Uniformly sample a fixed number of slices.
    """

    n = volume.shape[0]

    if n == num_slices:
        return volume

    indices = np.linspace(
        0,
        n - 1,
        num_slices
    ).round().astype(int)

    return volume[indices]


def normalize_slice(image):
    """
    Robust per-slice percentile normalization.

    Output approximately [0, 1].
    """

    image = image.astype(
        np.float32
    )

    low = np.percentile(
        image,
        1
    )

    high = np.percentile(
        image,
        99
    )

    if high <= low:
        return np.zeros_like(
            image,
            dtype=np.float32
        )

    image = np.clip(
        image,
        low,
        high
    )

    image = (
        image - low
    ) / (
        high - low
    )

    return image.astype(
        np.float32
    )


def preprocess_volume(
    volume,
    num_slices=16,
    image_size=224
):
    """
    Convert [N,H,W] volume into
    [N,1,image_size,image_size].
    """

    volume = sample_slices(
        volume,
        num_slices=num_slices
    )

    processed = []

    for image in volume:

        image = normalize_slice(
            image
        )

        tensor = torch.from_numpy(
            image
        ).unsqueeze(0).unsqueeze(0)

        tensor = F.interpolate(
            tensor,
            size=(image_size, image_size),
            mode="bilinear",
            align_corners=False
        )

        processed.append(
            tensor.squeeze(0)
        )

    return torch.stack(
        processed,
        dim=0
    )


class KneeStudyDataset(Dataset):

    def __init__(
        self,
        dataframe,
        root,
        num_slices=16,
        image_size=224,
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.root = Path(root)

        self.num_slices = num_slices
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        study_id = row[
            "StudyInstanceUID"
        ]

        sequence_tensors = []
        availability = []

        for sequence in SEQUENCES:

            series_id = row[
                sequence
            ]

            if pd.isna(series_id):

                # Missing sequence
                tensor = torch.zeros(
                    self.num_slices,
                    1,
                    self.image_size,
                    self.image_size,
                    dtype=torch.float32
                )

                availability.append(0)

            else:

                series_path = (
                    self.root
                    / "train_series"
                    / str(study_id)
                    / str(series_id)
                )

                volume = load_dicom_volume(
                    series_path
                )

                tensor = preprocess_volume(
                    volume,
                    num_slices=self.num_slices,
                    image_size=self.image_size
                )

                availability.append(1)

            sequence_tensors.append(
                tensor
            )

        labels = torch.tensor(
            row[TARGETS].values.astype(
                np.float32
            ),
            dtype=torch.float32
        )

        return {
            "study_id": study_id,

            "images": torch.stack(
                sequence_tensors,
                dim=0
            ),

            "availability": torch.tensor(
                availability,
                dtype=torch.float32
            ),

            "labels": labels,
        }

In [ ]:
dataset = KneeStudyDataset(
    dataframe=metadata,
    root=ROOT,
    num_slices=16,
    image_size=224
)

sample = dataset[0]

print("Study ID:")
print(sample["study_id"])

print("\nImages shape:")
print(sample["images"].shape)

print("\nAvailability:")
print(sample["availability"])

print("\nLabels:")
print(sample["labels"])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample = dataset[0]

images = sample["images"]
availability = sample["availability"]

sequence_names = [
    "Axial F1/FS1",
    "Sagittal F0/FS0",
    "Coronal F1/FS1",
    "Sagittal F1/FS1",
]

# Pick representative slices
slice_indices = [2, 6, 10, 13]

fig, axes = plt.subplots(
    4,
    4,
    figsize=(12, 12)
)

for row_idx, sequence_name in enumerate(sequence_names):

    for col_idx, slice_idx in enumerate(slice_indices):

        ax = axes[row_idx, col_idx]

        if availability[row_idx] == 0:
            ax.text(
                0.5,
                0.5,
                "MISSING",
                ha="center",
                va="center",
                fontsize=14
            )
            ax.axis("off")
            continue

        image = (
            images[row_idx, slice_idx, 0]
            .numpy()
        )

        ax.imshow(
            image,
            cmap="gray"
        )

        ax.set_title(
            f"{sequence_name}\nSlice {slice_idx + 1}"
        )

        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Full dataset preprocessing validation

print("Testing all 58 labelled studies...\n")

failed = []

for idx in range(len(dataset)):

    try:
        sample = dataset[idx]

        images = sample["images"]
        availability = sample["availability"]
        labels = sample["labels"]

        # Expected tensor structure
        assert images.shape == (
            4, 16, 1, 224, 224
        )

        assert availability.shape == (4,)
        assert labels.shape == (12,)

        # Check numerical validity
        assert torch.isfinite(images).all()
        assert torch.isfinite(labels).all()

    except Exception as e:

        failed.append({
            "index": idx,
            "study_id": metadata.iloc[idx]["StudyInstanceUID"],
            "error": str(e)
        })

    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(dataset)}")

print("\n" + "=" * 60)
print("VALIDATION RESULT")
print("=" * 60)

print("Total studies:", len(dataset))
print("Successful:", len(dataset) - len(failed))
print("Failed:", len(failed))

if failed:
    print("\nFailed studies:")
    display(pd.DataFrame(failed))
else:
    print("\nALL STUDIES PASSED.")

In [ ]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Multilabel-aware greedy 5-fold split
# ---------------------------------------------------------

targets = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

df = metadata.copy().reset_index(drop=True)

Y = df[targets].values.astype(int)

N_FOLDS = 5
RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)

n_samples = len(df)

# Random starting order, then prioritize rare labels
sample_order = np.arange(n_samples)
rng.shuffle(sample_order)

label_frequency = Y.sum(axis=0)

# Rare-label priority
sample_priority = np.zeros(n_samples)

for i in range(n_samples):
    positive_labels = np.where(Y[i] == 1)[0]

    if len(positive_labels) > 0:
        sample_priority[i] = sum(
            1.0 / max(label_frequency[j], 1)
            for j in positive_labels
        )

# Sort from rarest / most informative samples first
sample_order = sorted(
    sample_order,
    key=lambda i: sample_priority[i],
    reverse=True
)

fold_indices = [[] for _ in range(N_FOLDS)]
fold_label_counts = np.zeros(
    (N_FOLDS, len(targets)),
    dtype=float
)

fold_sizes = np.zeros(
    N_FOLDS,
    dtype=int
)

# Assign each sample to the fold currently most in need
for idx in sample_order:

    labels = Y[idx]

    best_fold = None
    best_score = None

    for fold in range(N_FOLDS):

        # Prefer folds with fewer samples
        size_penalty = fold_sizes[fold]

        # Prefer folds with fewer examples of this sample's labels
        label_penalty = np.sum(
            labels * fold_label_counts[fold]
        )

        score = (
            label_penalty * 10
            + size_penalty
        )

        if best_score is None or score < best_score:
            best_score = score
            best_fold = fold

    fold_indices[best_fold].append(idx)

    fold_label_counts[best_fold] += labels
    fold_sizes[best_fold] += 1


# Add fold assignments to metadata
fold_assignment = np.full(
    n_samples,
    -1,
    dtype=int
)

for fold, indices in enumerate(fold_indices):
    fold_assignment[indices] = fold

df["fold"] = fold_assignment

print("Fold sizes:")
print(
    df["fold"]
    .value_counts()
    .sort_index()
)

print("\nLabel distribution by fold:")

for fold in range(N_FOLDS):

    subset = df[df["fold"] == fold]

    distribution = pd.DataFrame({
        "positive": subset[targets].sum(),
        "total": len(subset),
        "positive_rate_%":
            (subset[targets].mean() * 100).round(1)
    })

    print(f"\n{'='*50}")
    print(f"FOLD {fold}")
    print(f"{'='*50}")

    display(distribution)

In [ ]:
# Compact 5-fold validation summary

fold_summary = []

for fold in range(N_FOLDS):
    subset = df[df["fold"] == fold]

    row = {
        "fold": fold,
        "n_studies": len(subset)
    }

    for target in targets:
        row[target] = int(subset[target].sum())

    fold_summary.append(row)

fold_summary = pd.DataFrame(fold_summary)

print("Positive cases in each validation fold:")
display(fold_summary)

print("\nMinimum positives available for each target across folds:")

min_positive = fold_summary[targets].min().sort_values()

display(
    min_positive.to_frame("minimum_positive_cases_in_any_fold")
)

print("\nFold sizes:")
display(
    fold_summary[["fold", "n_studies"]]
)

In [ ]:
print("Positive cases in each fold:")

for fold in range(5):
    subset = df[df["fold"] == fold]

    print(f"\nFOLD {fold} — {len(subset)} studies")

    for target in targets:
        print(
            f"{target:20s}: "
            f"{int(subset[target].sum())}"
        )

In [ ]:
# Freeze Baseline V1 folds

fold_path = "/kaggle/working/study_metadata_v2_folds.csv"

df.to_csv(
    fold_path,
    index=False
)

print("Saved frozen CV split:")
print(fold_path)

print("\nFold counts:")
print(
    df["fold"]
    .value_counts()
    .sort_index()
)

In [ ]:
from pathlib import Path
import torch
import pandas as pd
import time

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

cache_root = Path("/kaggle/working/knee_cache")
cache_root.mkdir(parents=True, exist_ok=True)

folded_metadata = pd.read_csv(
    "/kaggle/working/study_metadata_v2_folds.csv"
)

print("Studies to cache:", len(folded_metadata))
print("Cache directory:", cache_root)


# ---------------------------------------------------------
# Cache every labelled study
# ---------------------------------------------------------

cache_results = []

start_time = time.time()

for idx in range(len(folded_metadata)):

    row = folded_metadata.iloc[idx]
    study_id = row["StudyInstanceUID"]

    output_path = cache_root / f"{study_id}.pt"

    # Skip if already cached
    if output_path.exists():

        cache_results.append({
            "study_id": study_id,
            "cached": True,
            "status": "already_exists"
        })

        continue

    try:

        # Use our verified Dataset implementation
        sample = dataset[idx]

        torch.save(
            {
                "study_id": sample["study_id"],
                "images": sample["images"].to(
                    torch.float32
                ),
                "availability": sample["availability"].to(
                    torch.float32
                ),
                "labels": sample["labels"].to(
                    torch.float32
                ),
                "fold": int(row["fold"]),
            },
            output_path
        )

        cache_results.append({
            "study_id": study_id,
            "cached": True,
            "status": "created"
        })

    except Exception as e:

        cache_results.append({
            "study_id": study_id,
            "cached": False,
            "status": str(e)
        })

    if (idx + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(
            f"Processed {idx + 1}/{len(folded_metadata)} "
            f"| elapsed: {elapsed:.1f}s"
        )


cache_results = pd.DataFrame(cache_results)

print("\n" + "=" * 60)
print("CACHE RESULT")
print("=" * 60)

display(
    cache_results["status"]
    .value_counts()
    .to_frame("count")
)

failed = cache_results[
    ~cache_results["cached"]
]

print("\nFailed:", len(failed))

if len(failed):
    display(failed)

In [ ]:
# Verify cached tensors

files = sorted(cache_root.glob("*.pt"))

print("Cached studies:", len(files))

sample_file = files[0]

sample_cache = torch.load(
    sample_file,
    map_location="cpu"
)

print("\nSample cache contents:")

for key, value in sample_cache.items():

    if torch.is_tensor(value):
        print(
            key,
            "shape =", tuple(value.shape),
            "dtype =", value.dtype
        )
    else:
        print(
            key,
            "=",
            value
        )

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models


class BaselineV1(nn.Module):

    def __init__(
        self,
        num_classes=12,
        num_sequences=4,
        pretrained=True,
    ):
        super().__init__()

        weights = (
            models.ResNet18_Weights.DEFAULT
            if pretrained
            else None
        )

        backbone = models.resnet18(
            weights=weights
        )

        feature_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.encoder = backbone
        self.feature_dim = feature_dim
        self.num_sequences = num_sequences

        fusion_dim = (
            num_sequences * feature_dim
            + num_sequences
        )

        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.30),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.20),
        )

        self.classifier = nn.Linear(
            256,
            num_classes
        )

    def forward(self, x, availability):

        B, S, N, C, H, W = x.shape

        x = x.reshape(
            B * S * N,
            C,
            H,
            W
        )

        # Grayscale → RGB
        x = x.repeat(1, 3, 1, 1)

        features = self.encoder(x)

        features = features.reshape(
            B,
            S,
            N,
            self.feature_dim
        )

        # Slice pooling
        sequence_features = features.mean(dim=2)

        # Explicitly remove missing-sequence contribution
        sequence_features = (
            sequence_features
            * availability.unsqueeze(-1)
        )

        sequence_features = sequence_features.reshape(
            B,
            S * self.feature_dim
        )

        fused = torch.cat(
            [
                sequence_features,
                availability
            ],
            dim=1
        )

        fused = self.fusion(fused)

        logits = self.classifier(fused)

        return logits

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

model = BaselineV1(
    num_classes=12,
    num_sequences=4,
    pretrained=True
).to(device)

print("\nModel created successfully.")

# Load one cached study
cache_file = sorted(cache_root.glob("*.pt"))[0]

sample_cache = torch.load(
    cache_file,
    map_location="cpu",
    weights_only=False
)

images = sample_cache["images"].unsqueeze(0).to(device)
availability = sample_cache["availability"].unsqueeze(0).to(device)

print("\nInput shape:", images.shape)
print("Availability shape:", availability.shape)

with torch.no_grad():
    logits = model(
        images,
        availability
    )

print("\nOutput shape:", logits.shape)
print("Output:", logits)

In [ ]:
model = BaselineV1(
    num_classes=12,
    num_sequences=4,
    pretrained=True
).to(device)

with torch.no_grad():
    logits = model(
        images,
        availability
    )

print("Input:", images.shape)
print("Output:", logits.shape)
print(logits)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from pathlib import Path
import numpy as np


# =========================================================
# 1. Cached dataset
# =========================================================

class CachedKneeDataset(Dataset):

    def __init__(self, cache_files):
        self.files = list(cache_files)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        data = torch.load(
            self.files[idx],
            map_location="cpu",
            weights_only=False
        )

        return {
            "images": data["images"].float(),
            "availability": data["availability"].float(),
            "labels": data["labels"].float(),
            "study_id": data["study_id"],
            "fold": int(data["fold"])
        }


cache_files = sorted(
    Path("/kaggle/working/knee_cache").glob("*.pt")
)

all_cached = CachedKneeDataset(cache_files)

print("Cached studies:", len(all_cached))

In [ ]:
fold = 0

train_files = []
valid_files = []

for file in cache_files:

    data = torch.load(
        file,
        map_location="cpu",
        weights_only=False
    )

    if int(data["fold"]) == fold:
        valid_files.append(file)
    else:
        train_files.append(file)

print("Fold:", fold)
print("Training studies:", len(train_files))
print("Validation studies:", len(valid_files))

train_dataset = CachedKneeDataset(train_files)
valid_dataset = CachedKneeDataset(valid_files)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(valid_loader))

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = BaselineV1(
    num_classes=12,
    num_sequences=4,
    pretrained=True
).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("Device:", device)
print("Model parameters:",
      sum(p.numel() for p in model.parameters()))

In [ ]:
model.train()

running_loss = 0.0

for batch_idx, batch in enumerate(train_loader):

    images = batch["images"].to(
        device,
        non_blocking=True
    )

    availability = batch["availability"].to(
        device,
        non_blocking=True
    )

    labels = batch["labels"].to(
        device,
        non_blocking=True
    )

    optimizer.zero_grad()

    logits = model(
        images,
        availability
    )

    loss = criterion(
        logits,
        labels
    )

    loss.backward()

    optimizer.step()

    running_loss += loss.item()

    print(
        f"Batch {batch_idx + 1}/{len(train_loader)} "
        f"| loss = {loss.item():.4f}"
    )

print(
    "\nAverage training loss:",
    running_loss / len(train_loader)
)

In [ ]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in valid_loader:

        images = batch["images"].to(device)
        availability = batch["availability"].to(device)

        logits = model(
            images,
            availability
        )

        probabilities = torch.sigmoid(
            logits
        )

        all_predictions.append(
            probabilities.cpu().numpy()
        )

        all_labels.append(
            batch["labels"].numpy()
        )

predictions = np.concatenate(
    all_predictions,
    axis=0
)

labels = np.concatenate(
    all_labels,
    axis=0
)

print("Prediction shape:", predictions.shape)
print("Label shape:", labels.shape)

In [ ]:
auc_results = {}

for i, target in enumerate(targets):

    y_true = labels[:, i]
    y_pred = predictions[:, i]

    # AUC is undefined if validation contains only one class
    if len(np.unique(y_true)) < 2:
        auc_results[target] = np.nan
        continue

    auc_results[target] = roc_auc_score(
        y_true,
        y_pred
    )

auc_table = pd.DataFrame(
    {
        "target": list(auc_results.keys()),
        "auc": list(auc_results.values())
    }
)

display(auc_table)

macro_auc = np.nanmean(
    list(auc_results.values())
)

print("\nFold 0 Macro ROC-AUC:", macro_auc)

In [ ]:
# ============================================================
# RSNA Knee Abnormality Detection
# BASELINE V1 — Frozen ResNet18 + Study-Level Classifier
#
# Pipeline:
# cached MRI tensors
#     ↓
# ImageNet-pretrained ResNet18 (FROZEN)
#     ↓
# 16 slice embeddings → mean pooling
#     ↓
# 4 sequence embeddings
#     ↓
# availability mask
#     ↓
# study-level logistic regression
#     ↓
# 5-fold OOF predictions
#     ↓
# macro ROC-AUC
# ============================================================

import os
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


# ------------------------------------------------------------
# 0. Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ------------------------------------------------------------
# 1. Paths and configuration
# ------------------------------------------------------------

CACHE_ROOT = Path(
    "/kaggle/working/knee_cache"
)

METADATA_PATH = Path(
    "/kaggle/working/study_metadata_v2_folds.csv"
)

OUTPUT_ROOT = Path(
    "/kaggle/working/baseline_v1"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

NUM_SEQUENCES = 4
NUM_SLICES = 16
FEATURE_DIM = 512

SEQUENCES = [
    "axial_f1_fs1",
    "sagittal_f0_fs0",
    "coronal_f1_fs1",
    "sagittal_f1_fs1",
]

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]


# ------------------------------------------------------------
# 2. Load metadata
# ------------------------------------------------------------

metadata = pd.read_csv(
    METADATA_PATH
).reset_index(drop=True)

print(
    "\nNumber of labelled studies:",
    len(metadata)
)

print(
    "Fold counts:"
)

print(
    metadata["fold"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 3. Load pretrained ResNet18
# ------------------------------------------------------------

print("\nLoading pretrained ResNet18...")

weights = models.ResNet18_Weights.DEFAULT

backbone = models.resnet18(
    weights=weights
)

backbone.fc = nn.Identity()

backbone = backbone.to(device)

backbone.eval()

# IMPORTANT:
# We are freezing the image encoder.
for parameter in backbone.parameters():
    parameter.requires_grad = False

feature_extractor_preprocess = weights.transforms()

print(
    "Feature dimension:",
    FEATURE_DIM
)


# ------------------------------------------------------------
# 4. Extract frozen image features
# ------------------------------------------------------------
#
# Each study becomes:
#
#   sequence 1 → 512
#   sequence 2 → 512
#   sequence 3 → 512
#   sequence 4 → 512
#   availability → 4
#
# Final vector:
#
#   512*4 + 4 = 2052 dimensions
#
# ------------------------------------------------------------

embedding_path = (
    OUTPUT_ROOT / "study_embeddings.pt"
)

if embedding_path.exists():

    print(
        "\nExisting embedding file found."
    )

    embedding_data = torch.load(
        embedding_path,
        map_location="cpu",
        weights_only=False
    )

    embeddings = embedding_data["embeddings"]
    availability_matrix = embedding_data[
        "availability"
    ]
    study_ids = embedding_data[
        "study_ids"
    ]

    print(
        "Loaded cached embeddings:",
        embeddings.shape
    )

else:

    print(
        "\nExtracting frozen image embeddings..."
    )

    study_embeddings = []
    study_availability = []
    study_ids = []

    cache_files = sorted(
        CACHE_ROOT.glob("*.pt")
    )

    # Map study ID → cache file
    cache_map = {}

    for file_path in cache_files:

        cache_map[
            file_path.stem
        ] = file_path

    start_time = time.time()

    with torch.inference_mode():

        for index, row in metadata.iterrows():

            study_id = row[
                "StudyInstanceUID"
            ]

            cache_file = cache_map.get(
                study_id
            )

            if cache_file is None:
                raise FileNotFoundError(
                    f"No cache found for {study_id}"
                )

            sample = torch.load(
                cache_file,
                map_location="cpu",
                weights_only=False
            )

            images = sample[
                "images"
            ].float()

            availability = sample[
                "availability"
            ].float()

            # ------------------------------------------------
            # images shape:
            # [4, 16, 1, 224, 224]
            # ------------------------------------------------

            sequence_features = []

            for sequence_idx in range(
                NUM_SEQUENCES
            ):

                if (
                    availability[
                        sequence_idx
                    ].item() == 0
                ):
                    # Missing sequence:
                    # use an all-zero feature vector.
                    sequence_feature = torch.zeros(
                        FEATURE_DIM,
                        dtype=torch.float32
                    )

                else:

                    sequence_images = images[
                        sequence_idx
                    ]

                    # [16, 1, 224, 224]
                    #
                    # Repeat grayscale → RGB
                    sequence_images = (
                        sequence_images
                        .repeat(1, 3, 1, 1)
                    )

                    # ImageNet normalization
                    processed = []

                    for image in sequence_images:

                        image = (
                            image * 255.0
                        ).clamp(
                            0,
                            255
                        )

                        image = (
                            image / 255.0
                        )

                        image = (
                            image
                        )

                        image = (
                            image
                        )

                        image = (
                            image
                        )

                        # Normalize with
                        # ImageNet statistics
                        image = (
                            image
                            - torch.tensor(
                                [0.485, 0.456, 0.406]
                            ).view(3, 1, 1)
                        ) / torch.tensor(
                            [0.229, 0.224, 0.225]
                        ).view(3, 1, 1)

                        processed.append(
                            image
                        )

                    sequence_images = torch.stack(
                        processed,
                        dim=0
                    )

                    # Process all 16 slices
                    # together through ResNet.
                    sequence_images = (
                        sequence_images
                        .to(device)
                    )

                    features = backbone(
                        sequence_images
                    )

                    # [16, 512]
                    #
                    # Study-level representation
                    # for this sequence.
                    sequence_feature = (
                        features.mean(
                            dim=0
                        )
                        .cpu()
                    )

                sequence_features.append(
                    sequence_feature
                )

            # [4, 512]
            sequence_features = torch.stack(
                sequence_features
            )

            # Flatten:
            # [2048]
            sequence_features = (
                sequence_features
                .flatten()
            )

            # Add availability:
            # [2052]
            final_embedding = torch.cat(
                [
                    sequence_features,
                    availability
                ]
            )

            study_embeddings.append(
                final_embedding
            )

            study_availability.append(
                availability
            )

            study_ids.append(
                study_id
            )

            if (
                (index + 1) % 10 == 0
                or index == len(metadata) - 1
            ):

                elapsed = (
                    time.time()
                    - start_time
                )

                print(
                    f"Processed "
                    f"{index + 1}/{len(metadata)} "
                    f"| elapsed "
                    f"{elapsed:.1f}s"
                )

    embeddings = torch.stack(
        study_embeddings
    )

    availability_matrix = torch.stack(
        study_availability
    )

    torch.save(
        {
            "embeddings": embeddings,
            "availability": availability_matrix,
            "study_ids": study_ids,
        },
        embedding_path
    )

    print(
        "\nSaved embeddings:",
        embedding_path
    )


print(
    "\nEmbedding shape:",
    tuple(embeddings.shape)
)


# ------------------------------------------------------------
# 5. Prepare labels and folds
# ------------------------------------------------------------

X = embeddings.numpy().astype(
    np.float32
)

Y = metadata[
    TARGETS
].values.astype(
    np.float32
)

folds = metadata[
    "fold"
].values.astype(
    int
)


# ------------------------------------------------------------
# 6. Five-fold OOF training
# ------------------------------------------------------------
#
# We use a regularized logistic regression because:
#
# - only 58 labelled studies exist
# - feature dimension is high
# - a large neural classifier would overfit easily
# - the resulting model is fast and reproducible
#
# ------------------------------------------------------------

oof_predictions = np.zeros_like(
    Y,
    dtype=np.float32
)

fold_results = []

print(
    "\n"
    + "=" * 70
)

print(
    "STARTING 5-FOLD BASELINE V1"
)

print(
    "=" * 70
)


for fold in range(5):

    print(
        f"\n{'=' * 70}"
    )

    print(
        f"FOLD {fold}"
    )

    print(
        f"{'=' * 70}"
    )

    train_idx = np.where(
        folds != fold
    )[0]

    valid_idx = np.where(
        folds == fold
    )[0]

    X_train = X[
        train_idx
    ]

    X_valid = X[
        valid_idx
    ]

    print(
        "Train studies:",
        len(train_idx)
    )

    print(
        "Validation studies:",
        len(valid_idx)
    )

    fold_auc = []

    for target_idx, target in enumerate(
        TARGETS
    ):

        y_train = Y[
            train_idx,
            target_idx
        ]

        y_valid = Y[
            valid_idx,
            target_idx
        ]

        # ----------------------------------------------------
        # Regularized classifier
        # ----------------------------------------------------

        model = Pipeline(
            [
                (
                    "scaler",
                    StandardScaler()
                ),
                (
                    "classifier",
                    LogisticRegression(
                        C=0.1,
                        class_weight="balanced",
                        max_iter=3000,
                        solver="liblinear",
                        random_state=SEED
                    )
                )
            ]
        )

        model.fit(
            X_train,
            y_train
        )

        probabilities = model.predict_proba(
            X_valid
        )[:, 1]

        oof_predictions[
            valid_idx,
            target_idx
        ] = probabilities

        # AUC should be valid because
        # we explicitly checked that each
        # target has positives in every fold.
        auc = roc_auc_score(
            y_valid,
            probabilities
        )

        fold_auc.append(
            auc
        )

        print(
            f"{target:20s} "
            f"AUC = {auc:.4f}"
        )

    macro_auc = float(
        np.mean(fold_auc)
    )

    fold_results.append(
        {
            "fold": fold,
            "macro_auc": macro_auc,
            "n_train": len(train_idx),
            "n_valid": len(valid_idx)
        }
    )

    print(
        f"\nFold {fold} Macro AUC: "
        f"{macro_auc:.4f}"
    )


# ------------------------------------------------------------
# 7. Overall OOF score
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 70
)

print(
    "OVERALL OOF RESULTS"
)

print(
    "=" * 70
)


oof_auc = {}

for target_idx, target in enumerate(
    TARGETS
):

    auc = roc_auc_score(
        Y[:, target_idx],
        oof_predictions[:, target_idx]
    )

    oof_auc[target] = auc


results_table = pd.DataFrame(
    {
        "target": TARGETS,
        "OOF_AUC": [
            oof_auc[t]
            for t in TARGETS
        ]
    }
)

results_table = results_table.sort_values(
    "OOF_AUC",
    ascending=False
).reset_index(
    drop=True
)

display(
    results_table
)

overall_macro_auc = float(
    results_table["OOF_AUC"].mean()
)

print(
    "\nOverall OOF Macro ROC-AUC:",
    f"{overall_macro_auc:.6f}"
)


# ------------------------------------------------------------
# 8. Fold summary
# ------------------------------------------------------------

fold_results_df = pd.DataFrame(
    fold_results
)

print(
    "\nFold-level results:"
)

display(
    fold_results_df
)


print(
    "\nMean fold Macro AUC:",
    f"{fold_results_df['macro_auc'].mean():.6f}"
)

print(
    "Std fold Macro AUC:",
    f"{fold_results_df['macro_auc'].std(ddof=1):.6f}"
)


# ------------------------------------------------------------
# 9. Save predictions/results
# ------------------------------------------------------------

oof_df = pd.DataFrame(
    oof_predictions,
    columns=[
        f"{target}_prediction"
        for target in TARGETS
    ]
)

oof_df.insert(
    0,
    "StudyInstanceUID",
    metadata[
        "StudyInstanceUID"
    ].values
)

oof_df["fold"] = folds

oof_path = (
    OUTPUT_ROOT
    / "oof_predictions.csv"
)

oof_df.to_csv(
    oof_path,
    index=False
)

results_path = (
    OUTPUT_ROOT
    / "baseline_v1_results.csv"
)

results_table.to_csv(
    results_path,
    index=False
)

fold_results_path = (
    OUTPUT_ROOT
    / "fold_results.csv"
)

fold_results_df.to_csv(
    fold_results_path,
    index=False
)


# ------------------------------------------------------------
# 10. Final report
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 70
)

print(
    "BASELINE V1 COMPLETE"
)

print(
    "=" * 70
)

print(
    "Overall OOF Macro ROC-AUC:",
    f"{overall_macro_auc:.6f}"
)

print(
    "\nSaved:"
)

print(
    "Embeddings:",
    embedding_path
)

print(
    "OOF predictions:",
    oof_path
)

print(
    "Target results:",
    results_path
)

print(
    "Fold results:",
    fold_results_path
)

In [ ]:
results = pd.read_csv(
    "/kaggle/working/baseline_v1/baseline_v1_results.csv"
)

display(
    results.sort_values(
        "OOF_AUC",
        ascending=False
    )
)

In [ ]:
# ============================================================
# RSNA Knee Abnormality Detection
# BASELINE V2 — Target-Aware Slice Attention
#
# V1:
#   16 slice embeddings -> MEAN -> sequence feature
#
# V2:
#   16 slice embeddings -> target-aware ATTENTION -> sequence feature
#
# The ResNet-18 image encoder remains frozen.
# We keep the exact same 5 folds as V1.
# ============================================================

import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models

from sklearn.metrics import roc_auc_score


# ============================================================
# 0. Configuration
# ============================================================

SEED = 42
NUM_FOLDS = 5

NUM_SEQUENCES = 4
NUM_SLICES = 16
FEATURE_DIM = 512
NUM_TARGETS = 12

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CACHE_ROOT = Path(
    "/kaggle/working/knee_cache"
)

METADATA_PATH = Path(
    "/kaggle/working/study_metadata_v2_folds.csv"
)

OUTPUT_ROOT = Path(
    "/kaggle/working/baseline_v2"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

SEQUENCES = [
    "axial_f1_fs1",
    "sagittal_f0_fs0",
    "coronal_f1_fs1",
    "sagittal_f1_fs1",
]

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]


# ============================================================
# 1. Reproducibility
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)


# ============================================================
# 2. Load fixed metadata / folds
# ============================================================

metadata = pd.read_csv(
    METADATA_PATH
).reset_index(drop=True)

Y = metadata[TARGETS].values.astype(
    np.float32
)

FOLDS = metadata["fold"].values.astype(
    int
)

print("\nStudies:", len(metadata))
print("\nFrozen fold sizes:")
print(
    metadata["fold"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 3. Frozen ResNet-18
# ============================================================

print("\nLoading pretrained ResNet-18...")

weights = models.ResNet18_Weights.DEFAULT

backbone = models.resnet18(
    weights=weights
)

backbone.fc = nn.Identity()

backbone = backbone.to(DEVICE)
backbone.eval()

for p in backbone.parameters():
    p.requires_grad = False

print("Backbone feature dimension:", FEATURE_DIM)


# ============================================================
# 4. Extract PER-SLICE embeddings
#
# Saved structure:
#
# [58, 4, 16, 512]
#
# 58 studies
# 4 sequences
# 16 slices
# 512 features per slice
# ============================================================

slice_embedding_path = (
    OUTPUT_ROOT / "slice_embeddings.pt"
)

if slice_embedding_path.exists():

    print(
        "\nLoading existing slice embeddings..."
    )

    saved = torch.load(
        slice_embedding_path,
        map_location="cpu",
        weights_only=False
    )

    slice_embeddings = saved[
        "slice_embeddings"
    ]

    availability = saved[
        "availability"
    ]

    print(
        "Loaded:",
        tuple(slice_embeddings.shape)
    )

else:

    print(
        "\nExtracting frozen per-slice embeddings..."
    )

    cache_map = {
        p.stem: p
        for p in CACHE_ROOT.glob("*.pt")
    }

    all_embeddings = []
    all_availability = []

    extraction_start = time.time()

    with torch.inference_mode():

        for study_idx, row in metadata.iterrows():

            study_id = row[
                "StudyInstanceUID"
            ]

            cache_file = cache_map.get(
                study_id
            )

            if cache_file is None:
                raise FileNotFoundError(
                    f"Cache missing: {study_id}"
                )

            sample = torch.load(
                cache_file,
                map_location="cpu",
                weights_only=False
            )

            images = sample[
                "images"
            ].float()

            study_availability = sample[
                "availability"
            ].float()

            sequence_features = []

            for seq_idx in range(
                NUM_SEQUENCES
            ):

                if study_availability[
                    seq_idx
                ].item() == 0:

                    sequence_features.append(
                        torch.zeros(
                            NUM_SLICES,
                            FEATURE_DIM,
                            dtype=torch.float32
                        )
                    )

                    continue

                sequence_images = images[
                    seq_idx
                ]

                # [16,1,224,224] -> [16,3,224,224]
                sequence_images = (
                    sequence_images
                    .repeat(1, 3, 1, 1)
                )

                # ImageNet normalization
                mean = torch.tensor(
                    [0.485, 0.456, 0.406],
                    dtype=torch.float32
                ).view(1, 3, 1, 1)

                std = torch.tensor(
                    [0.229, 0.224, 0.225],
                    dtype=torch.float32
                ).view(1, 3, 1, 1)

                sequence_images = (
                    sequence_images - mean
                ) / std

                sequence_images = (
                    sequence_images
                    .to(DEVICE)
                )

                features = backbone(
                    sequence_images
                )

                # [16,512]
                sequence_features.append(
                    features.cpu()
                )

            # [4,16,512]
            all_embeddings.append(
                torch.stack(
                    sequence_features
                )
            )

            all_availability.append(
                study_availability
            )

            if (
                (study_idx + 1) % 10 == 0
                or study_idx == len(metadata) - 1
            ):

                elapsed = (
                    time.time()
                    - extraction_start
                )

                print(
                    f"Processed "
                    f"{study_idx + 1}/"
                    f"{len(metadata)} "
                    f"| {elapsed:.1f}s"
                )

    slice_embeddings = torch.stack(
        all_embeddings
    )

    availability = torch.stack(
        all_availability
    )

    torch.save(
        {
            "slice_embeddings":
                slice_embeddings,
            "availability":
                availability,
        },
        slice_embedding_path
    )

    print(
        "\nSaved:",
        slice_embedding_path
    )

print(
    "Slice embedding shape:",
    tuple(slice_embeddings.shape)
)


# ============================================================
# 5. Target-Aware Attention Model
# ============================================================

class TargetAwareAttention(nn.Module):
    """
    For each target, learn a query vector.

    For each sequence:

        16 slice features
              ↓
        similarity with target query
              ↓
        softmax attention
              ↓
        weighted slice representation

    This lets ACL, MCL, OA, etc. focus on
    different slices.
    """

    def __init__(
        self,
        num_targets=12,
        num_sequences=4,
        feature_dim=512,
    ):
        super().__init__()

        self.num_targets = num_targets
        self.num_sequences = num_sequences
        self.feature_dim = feature_dim

        # Small target-specific queries.
        self.query = nn.Parameter(
            torch.randn(
                num_targets,
                feature_dim
            ) * 0.02
        )

        # Temperature prevents extremely sharp
        # attention at initialization.
        self.temperature = nn.Parameter(
            torch.ones(num_targets)
        )

        # One lightweight target-specific
        # classifier per target.
        self.heads = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(
                        num_sequences * feature_dim
                        + num_sequences,
                        128
                    ),
                    nn.LayerNorm(128),
                    nn.GELU(),
                    nn.Dropout(0.25),
                    nn.Linear(128, 1)
                )
                for _ in range(num_targets)
            ]
        )

    def forward(
        self,
        x,
        availability
    ):
        """
        x:
            [B, S, N, D]

        availability:
            [B, S]

        returns:
            logits [B, T]
        """

        B, S, N, D = x.shape

        target_outputs = []

        # ----------------------------------------------------
        # Each target gets its own attention mechanism.
        # ----------------------------------------------------

        for target_idx in range(
            self.num_targets
        ):

            query = self.query[
                target_idx
            ]

            temperature = (
                self.temperature[
                    target_idx
                ].abs()
                + 1e-4
            )

            # ------------------------------------------------
            # x:
            # [B,S,N,D]
            #
            # query:
            # [D]
            #
            # scores:
            # [B,S,N]
            # ------------------------------------------------

            scores = torch.einsum(
                "bsnd,d->bsn",
                x,
                query
            )

            scores = (
                scores
                / temperature
            )

            # ------------------------------------------------
            # Mask completely missing sequences.
            # ------------------------------------------------

            missing_mask = (
                availability == 0
            )

            scores = scores.masked_fill(
                missing_mask.unsqueeze(-1),
                -1e4
            )

            # ------------------------------------------------
            # Attention over slices
            # ------------------------------------------------

            attention = torch.softmax(
                scores,
                dim=2
            )

            # ------------------------------------------------
            # Weighted slice pooling
            # ------------------------------------------------

            pooled = torch.sum(
                x * attention.unsqueeze(-1),
                dim=2
            )

            # Remove representations from
            # missing sequences explicitly.
            pooled = (
                pooled
                * availability.unsqueeze(-1)
            )

            # [B,S,D] -> [B,S*D]
            pooled = pooled.reshape(
                B,
                S * D
            )

            fused = torch.cat(
                [
                    pooled,
                    availability
                ],
                dim=1
            )

            logits = self.heads[
                target_idx
            ](
                fused
            )

            target_outputs.append(
                logits
            )

        return torch.cat(
            target_outputs,
            dim=1
        )


# ============================================================
# 6. Training helpers
# ============================================================

def evaluate_model(
    model,
    X_valid,
    A_valid,
    y_valid
):

    model.eval()

    with torch.no_grad():

        logits = model(
            X_valid,
            A_valid
        )

        probabilities = (
            torch.sigmoid(
                logits
            )
            .cpu()
            .numpy()
        )

    y_true = (
        y_valid
        .cpu()
        .numpy()
    )

    aucs = []

    for target_idx in range(
        NUM_TARGETS
    ):

        auc = roc_auc_score(
            y_true[:, target_idx],
            probabilities[:, target_idx]
        )

        aucs.append(auc)

    return (
        np.asarray(aucs),
        probabilities
    )


# ============================================================
# 7. Five-fold training
# ============================================================

oof_predictions = np.zeros(
    (
        len(metadata),
        NUM_TARGETS
    ),
    dtype=np.float32
)

fold_summaries = []

print(
    "\n"
    + "=" * 70
)

print(
    "BASELINE V2 — TARGET-AWARE SLICE ATTENTION"
)

print(
    "=" * 70
)


for fold in range(
    NUM_FOLDS
):

    print(
        f"\n{'=' * 70}"
    )

    print(
        f"FOLD {fold}"
    )

    print(
        f"{'=' * 70}"
    )

    train_idx = np.where(
        FOLDS != fold
    )[0]

    valid_idx = np.where(
        FOLDS == fold
    )[0]

    X_train = (
        slice_embeddings[
            train_idx
        ]
        .to(DEVICE)
    )

    A_train = (
        availability[
            train_idx
        ]
        .to(DEVICE)
    )

    y_train = torch.tensor(
        Y[train_idx],
        dtype=torch.float32,
        device=DEVICE
    )

    X_valid = (
        slice_embeddings[
            valid_idx
        ]
        .to(DEVICE)
    )

    A_valid = (
        availability[
            valid_idx
        ]
        .to(DEVICE)
    )

    y_valid = torch.tensor(
        Y[valid_idx],
        dtype=torch.float32,
        device=DEVICE
    )

    model = TargetAwareAttention(
        num_targets=NUM_TARGETS,
        num_sequences=NUM_SEQUENCES,
        feature_dim=FEATURE_DIM
    ).to(DEVICE)

    # --------------------------------------------------------
    # Positive-class weighting computed ONLY from training fold.
    # --------------------------------------------------------

    pos = (
        Y[train_idx]
        .sum(axis=0)
    )

    neg = (
        len(train_idx)
        - pos
    )

    pos_weight = torch.tensor(
        neg / np.maximum(pos, 1),
        dtype=torch.float32,
        device=DEVICE
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-3,
        weight_decay=1e-3
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=120
    )

    best_auc = -np.inf
    best_state = None
    patience = 25
    epochs_without_improvement = 0

    for epoch in range(1, 121):

        model.train()

        optimizer.zero_grad()

        logits = model(
            X_train,
            A_train
        )

        loss = criterion(
            logits,
            y_train
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        # ----------------------------------------------------
        # Validation every epoch.
        # Dataset is tiny, so this is cheap.
        # ----------------------------------------------------

        valid_aucs, _ = evaluate_model(
            model,
            X_valid,
            A_valid,
            y_valid
        )

        macro_auc = float(
            np.mean(valid_aucs)
        )

        if macro_auc > best_auc:

            best_auc = macro_auc

            best_state = {
                k: v.detach()
                .cpu()
                .clone()
                for k, v in model.state_dict().items()
            }

            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1

        if (
            epoch == 1
            or epoch % 10 == 0
            or epoch == 120
        ):

            print(
                f"Epoch {epoch:03d} "
                f"| loss {loss.item():.4f} "
                f"| val macro AUC "
                f"{macro_auc:.4f} "
                f"| best "
                f"{best_auc:.4f}"
            )

        if (
            epochs_without_improvement
            >= patience
        ):
            print(
                f"Early stopping at epoch "
                f"{epoch}"
            )
            break

    # --------------------------------------------------------
    # Restore best epoch.
    # --------------------------------------------------------

    model.load_state_dict(
        best_state
    )

    final_aucs, final_predictions = (
        evaluate_model(
            model,
            X_valid,
            A_valid,
            y_valid
        )
    )

    oof_predictions[
        valid_idx
    ] = final_predictions

    fold_macro = float(
        np.mean(final_aucs)
    )

    fold_summaries.append(
        {
            "fold": fold,
            "n_train": len(train_idx),
            "n_valid": len(valid_idx),
            "best_macro_auc": fold_macro
        }
    )

    print(
        f"\nFold {fold} FINAL macro AUC:"
        f" {fold_macro:.6f}"
    )

    # Save fold model
    torch.save(
        {
            "model_state_dict":
                model.state_dict(),
            "fold": fold,
            "best_macro_auc":
                fold_macro
        },
        OUTPUT_ROOT
        / f"fold_{fold}_model.pt"
    )


# ============================================================
# 8. Overall OOF evaluation
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "BASELINE V2 — OOF RESULTS"
)

print(
    "=" * 70
)

overall_aucs = []

for target_idx, target in enumerate(
    TARGETS
):

    auc = roc_auc_score(
        Y[:, target_idx],
        oof_predictions[:, target_idx]
    )

    overall_aucs.append(
        auc
    )


results = pd.DataFrame(
    {
        "target": TARGETS,
        "OOF_AUC": overall_aucs
    }
).sort_values(
    "OOF_AUC",
    ascending=False
).reset_index(
    drop=True
)

display(
    results
)

overall_macro_auc = float(
    np.mean(overall_aucs)
)

print(
    "\nOverall OOF Macro ROC-AUC:",
    f"{overall_macro_auc:.6f}"
)


# ============================================================
# 9. Fold summary
# ============================================================

fold_results = pd.DataFrame(
    fold_summaries
)

print(
    "\nFold summary:"
)

display(
    fold_results
)

print(
    "\nMean fold Macro AUC:",
    f"{fold_results['best_macro_auc'].mean():.6f}"
)

print(
    "Std fold Macro AUC:",
    f"{fold_results['best_macro_auc'].std(ddof=1):.6f}"
)


# ============================================================
# 10. Compare against V1
# ============================================================

V1_AUC = 0.626641

print(
    "\n"
    + "=" * 70
)

print(
    "V1 vs V2"
)

print(
    "=" * 70
)

print(
    f"V1 Macro AUC: "
    f"{V1_AUC:.6f}"
)

print(
    f"V2 Macro AUC: "
    f"{overall_macro_auc:.6f}"
)

print(
    f"Absolute improvement: "
    f"{overall_macro_auc - V1_AUC:+.6f}"
)


# ============================================================
# 11. Save everything
# ============================================================

oof_output = pd.DataFrame(
    oof_predictions,
    columns=[
        f"{target}_prediction"
        for target in TARGETS
    ]
)

oof_output.insert(
    0,
    "StudyInstanceUID",
    metadata[
        "StudyInstanceUID"
    ].values
)

oof_output["fold"] = FOLDS

oof_output.to_csv(
    OUTPUT_ROOT
    / "oof_predictions.csv",
    index=False
)

results.to_csv(
    OUTPUT_ROOT
    / "baseline_v2_results.csv",
    index=False
)

fold_results.to_csv(
    OUTPUT_ROOT
    / "fold_results.csv",
    index=False
)

print(
    "\nSaved V2 results to:"
)

print(
    OUTPUT_ROOT
)

print(
    "\nBASELINE V2 COMPLETE."
)

In [ ]:
# ============================================================
# RSNA KNEE ABNORMALITY DETECTION
# FINAL V1 SUBMISSION PIPELINE
#
# Uses the BEST VALIDATED model so far:
#   Frozen ResNet18 embeddings
#   + mean slice pooling
#   + 4 sequence fusion
#   + availability mask
#   + one LogisticRegression classifier per target
#
# IMPORTANT:
# - Uses the SAME preprocessing as V1
# - Uses the SAME four sequence types
# - Trains final classifiers on all 58 labelled studies
# - Generates predictions for ALL test studies
# - Writes submission.csv
# ============================================================

import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pydicom
import torchvision.models as models

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ============================================================
# 1. Configuration
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

ROOT = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection"
)

TRAIN_PATH = ROOT / "train.csv"
TEST_PATH = ROOT / "test.csv"
TRAIN_SERIES_PATH = ROOT / "train_series.csv"
TEST_SERIES_PATH = ROOT / "test_series.csv"
SAMPLE_PATH = ROOT / "sample_submission.csv"

OUTPUT_DIR = Path(
    "/kaggle/working/final_submission_v1"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

SEQUENCE_TYPES = [
    ("Axial", 1, 1, "axial_f1_fs1"),
    ("Sagittal", 0, 0, "sagittal_f0_fs0"),
    ("Coronal", 1, 1, "coronal_f1_fs1"),
    ("Sagittal", 1, 1, "sagittal_f1_fs1"),
]

NUM_SLICES = 16
IMAGE_SIZE = 224
FEATURE_DIM = 512

print("Device:", DEVICE)


# ============================================================
# 2. Load competition tables
# ============================================================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
train_series = pd.read_csv(TRAIN_SERIES_PATH)
test_series = pd.read_csv(TEST_SERIES_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

print("\nTrain shape:", train.shape)
print("Test shape:", test.shape)
print("Train series shape:", train_series.shape)
print("Test series shape:", test_series.shape)
print("Sample submission shape:", sample_submission.shape)


# ============================================================
# 3. Select the 58 genuinely labelled studies
# ============================================================

labelled_train = train[
    train[TARGETS].notna().all(axis=1)
].copy()

print(
    "\nFully labelled studies:",
    len(labelled_train)
)

if len(labelled_train) != 58:
    raise RuntimeError(
        f"Expected 58 labelled studies, "
        f"found {len(labelled_train)}."
    )


# ============================================================
# 4. DICOM utilities
# ============================================================

def load_dicom_volume(series_path):
    """
    Load one DICOM series and order slices using
    ImageOrientationPatient + ImagePositionPatient.
    """

    series_path = Path(series_path)

    files = list(series_path.glob("*.dcm"))

    if not files:
        raise FileNotFoundError(
            f"No DICOM files found in {series_path}"
        )

    records = []

    for file_path in files:

        ds = pydicom.dcmread(
            file_path,
            stop_before_pixels=False
        )

        position = np.asarray(
            ds.ImagePositionPatient,
            dtype=np.float64
        )

        orientation = np.asarray(
            ds.ImageOrientationPatient,
            dtype=np.float64
        )

        row_direction = orientation[:3]
        column_direction = orientation[3:]

        normal = np.cross(
            row_direction,
            column_direction
        )

        physical_position = np.dot(
            position,
            normal
        )

        records.append(
            (
                physical_position,
                file_path
            )
        )

    records.sort(
        key=lambda x: x[0]
    )

    images = []

    for _, file_path in records:

        ds = pydicom.dcmread(
            file_path,
            stop_before_pixels=False
        )

        image = ds.pixel_array.astype(
            np.float32
        )

        images.append(image)

    return np.stack(
        images,
        axis=0
    )


def sample_slices(
    volume,
    num_slices=NUM_SLICES
):

    n = volume.shape[0]

    indices = np.linspace(
        0,
        n - 1,
        num_slices
    ).round().astype(int)

    return volume[indices]


def normalize_slice(image):

    image = image.astype(
        np.float32
    )

    low = np.percentile(
        image,
        1
    )

    high = np.percentile(
        image,
        99
    )

    if high <= low:
        return np.zeros_like(
            image,
            dtype=np.float32
        )

    image = np.clip(
        image,
        low,
        high
    )

    image = (
        image - low
    ) / (
        high - low
    )

    return image.astype(
        np.float32
    )


def preprocess_volume(volume):

    volume = sample_slices(
        volume,
        NUM_SLICES
    )

    processed = []

    for image in volume:

        image = normalize_slice(
            image
        )

        tensor = torch.from_numpy(
            image
        ).unsqueeze(0).unsqueeze(0)

        tensor = torch.nn.functional.interpolate(
            tensor,
            size=(
                IMAGE_SIZE,
                IMAGE_SIZE
            ),
            mode="bilinear",
            align_corners=False
        )

        processed.append(
            tensor.squeeze(0)
        )

    return torch.stack(
        processed
    )


# ============================================================
# 5. Build deterministic series selection
# ============================================================

def build_selected_series_table(
    series_df,
    study_ids
):

    rows = []

    study_id_set = set(
        study_ids
    )

    for plane, fluid, fat, name in SEQUENCE_TYPES:

        subset = series_df[
            (series_df["Anatomical_Plane"] == plane) &
            (series_df["Fluid_Sensitive"] == fluid) &
            (series_df["Fat_Suppression"] == fat) &
            (series_df["StudyInstanceUID"].isin(study_id_set))
        ].copy()

        for study_id, group in subset.groupby(
            "StudyInstanceUID"
        ):

            best_series = None
            best_count = -1

            for _, series_row in group.iterrows():

                series_id = series_row[
                    "SeriesInstanceUID"
                ]

                series_path = (
                    ROOT
                    / (
                        "test_series"
                        if series_df is test_series
                        else "train_series"
                    )
                    / str(study_id)
                    / str(series_id)
                )

                n_slices = len(
                    list(
                        series_path.glob(
                            "*.dcm"
                        )
                    )
                )

                if n_slices > best_count:

                    best_count = n_slices
                    best_series = series_id

            if best_series is not None:

                rows.append(
                    {
                        "StudyInstanceUID":
                            study_id,
                        "sequence_type":
                            name,
                        "SeriesInstanceUID":
                            best_series,
                        "num_slices":
                            best_count
                    }
                )

    selected = pd.DataFrame(rows)

    pivot = selected.pivot(
        index="StudyInstanceUID",
        columns="sequence_type",
        values="SeriesInstanceUID"
    ).reset_index()

    for _, _, _, name in SEQUENCE_TYPES:

        if name not in pivot.columns:
            pivot[name] = np.nan

    return pivot[
        [
            "StudyInstanceUID",
            "axial_f1_fs1",
            "sagittal_f0_fs0",
            "coronal_f1_fs1",
            "sagittal_f1_fs1"
        ]
    ]


print(
    "\nBuilding test study → series mapping..."
)

test_series_map = build_selected_series_table(
    test_series,
    test["StudyInstanceUID"].tolist()
)

print(
    "Test studies with mappings:",
    len(test_series_map)
)

display(
    test_series_map.head()
)


# ============================================================
# 6. Load frozen ResNet-18 feature extractor
# ============================================================

print(
    "\nLoading pretrained ResNet-18..."
)

weights = models.ResNet18_Weights.DEFAULT

backbone = models.resnet18(
    weights=weights
)

backbone.fc = nn.Identity()

backbone = backbone.to(
    DEVICE
)

backbone.eval()

for parameter in backbone.parameters():
    parameter.requires_grad = False


# ImageNet normalization
imagenet_mean = torch.tensor(
    [0.485, 0.456, 0.406],
    dtype=torch.float32,
    device=DEVICE
).view(
    1, 3, 1, 1
)

imagenet_std = torch.tensor(
    [0.229, 0.224, 0.225],
    dtype=torch.float32,
    device=DEVICE
).view(
    1, 3, 1, 1
)


# ============================================================
# 7. Extract final TRAIN embeddings
#
# We use the SAME representation as V1:
#
# 4 sequences × mean of 16 slice embeddings
# + 4 availability indicators
#
# Final dimension = 2052
# ============================================================

print(
    "\nExtracting final training embeddings..."
)

train_series_map = build_selected_series_table(
    train_series,
    labelled_train["StudyInstanceUID"].tolist()
)

train_embedding_rows = []

start = time.time()

with torch.inference_mode():

    for count, study_id in enumerate(
        labelled_train["StudyInstanceUID"]
    ):

        mapping = train_series_map[
            train_series_map["StudyInstanceUID"]
            == study_id
        ]

        if len(mapping) != 1:
            raise RuntimeError(
                f"Missing/duplicate mapping for "
                f"train study {study_id}"
            )

        mapping = mapping.iloc[0]

        sequence_features = []
        availability = []

        for sequence in [
            "axial_f1_fs1",
            "sagittal_f0_fs0",
            "coronal_f1_fs1",
            "sagittal_f1_fs1"
        ]:

            series_id = mapping[
                sequence
            ]

            if pd.isna(series_id):

                sequence_features.append(
                    torch.zeros(
                        FEATURE_DIM,
                        dtype=torch.float32
                    )
                )

                availability.append(0.0)
                continue

            series_path = (
                ROOT
                / "train_series"
                / str(study_id)
                / str(series_id)
            )

            volume = load_dicom_volume(
                series_path
            )

            images = preprocess_volume(
                volume
            )

            # [16,1,224,224] → RGB
            images = images.repeat(
                1, 3, 1, 1
            )

            images = (
                images
                .to(DEVICE)
            )

            images = (
                images - imagenet_mean
            ) / imagenet_std

            slice_features = backbone(
                images
            )

            study_feature = (
                slice_features
                .mean(dim=0)
                .cpu()
            )

            sequence_features.append(
                study_feature
            )

            availability.append(1.0)

        feature_vector = torch.cat(
            sequence_features
            + [
                torch.tensor(
                    availability,
                    dtype=torch.float32
                )
            ]
        )

        train_embedding_rows.append(
            feature_vector.numpy()
        )

        if (
            (count + 1) % 10 == 0
            or count == len(labelled_train) - 1
        ):

            elapsed = (
                time.time() - start
            )

            print(
                f"Train embeddings: "
                f"{count + 1}/"
                f"{len(labelled_train)} "
                f"| {elapsed:.1f}s"
            )

X_train = np.stack(
    train_embedding_rows
).astype(
    np.float32
)

Y_train = labelled_train[
    TARGETS
].values.astype(
    np.float32
)

print(
    "\nFinal train embedding shape:",
    X_train.shape
)


# ============================================================
# 8. Train FINAL classifiers on ALL 58 labelled studies
# ============================================================

print(
    "\nTraining final 12 target classifiers..."
)

final_models = []

for target_idx, target in enumerate(
    TARGETS
):

    y = Y_train[
        :,
        target_idx
    ]

    model = Pipeline(
        [
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    C=0.1,
                    class_weight="balanced",
                    max_iter=3000,
                    solver="liblinear",
                    random_state=SEED
                )
            )
        ]
    )

    model.fit(
        X_train,
        y
    )

    final_models.append(
        model
    )

    print(
        f"Trained: {target}"
    )


# ============================================================
# 9. Extract TEST embeddings
# ============================================================

print(
    "\nExtracting test embeddings..."
)

test_embedding_rows = []
test_ids = test[
    "StudyInstanceUID"
].tolist()

start = time.time()

with torch.inference_mode():

    for count, study_id in enumerate(
        test_ids
    ):

        mapping = test_series_map[
            test_series_map["StudyInstanceUID"]
            == study_id
        ]

        if len(mapping) != 1:
            raise RuntimeError(
                f"Missing/duplicate mapping "
                f"for test study {study_id}"
            )

        mapping = mapping.iloc[0]

        sequence_features = []
        availability = []

        for sequence in [
            "axial_f1_fs1",
            "sagittal_f0_fs0",
            "coronal_f1_fs1",
            "sagittal_f1_fs1"
        ]:

            series_id = mapping[
                sequence
            ]

            if pd.isna(series_id):

                sequence_features.append(
                    torch.zeros(
                        FEATURE_DIM,
                        dtype=torch.float32
                    )
                )

                availability.append(0.0)

                continue

            series_path = (
                ROOT
                / "test_series"
                / str(study_id)
                / str(series_id)
            )

            if not series_path.exists():

                raise FileNotFoundError(
                    f"Missing test series:\n"
                    f"{series_path}"
                )

            volume = load_dicom_volume(
                series_path
            )

            images = preprocess_volume(
                volume
            )

            images = images.repeat(
                1, 3, 1, 1
            )

            images = images.to(
                DEVICE
            )

            images = (
                images - imagenet_mean
            ) / imagenet_std

            slice_features = backbone(
                images
            )

            study_feature = (
                slice_features
                .mean(dim=0)
                .cpu()
            )

            sequence_features.append(
                study_feature
            )

            availability.append(
                1.0
            )

        feature_vector = torch.cat(
            sequence_features
            + [
                torch.tensor(
                    availability,
                    dtype=torch.float32
                )
            ]
        )

        test_embedding_rows.append(
            feature_vector.numpy()
        )

        if (
            (count + 1) % 25 == 0
            or count == len(test_ids) - 1
        ):

            elapsed = (
                time.time() - start
            )

            print(
                f"Test embeddings: "
                f"{count + 1}/"
                f"{len(test_ids)} "
                f"| {elapsed:.1f}s"
            )

X_test = np.stack(
    test_embedding_rows
).astype(
    np.float32
)

print(
    "\nFinal test embedding shape:",
    X_test.shape
)


# ============================================================
# 10. Generate FINAL predictions
# ============================================================

predictions = {}

for target_idx, target in enumerate(
    TARGETS
):

    probabilities = (
        final_models[target_idx]
        .predict_proba(
            X_test
        )[:, 1]
    )

    probabilities = np.clip(
        probabilities,
        1e-6,
        1 - 1e-6
    )

    predictions[target] = probabilities


prediction_df = pd.DataFrame(
    predictions
)

prediction_df.insert(
    0,
    "StudyInstanceUID",
    test_ids
)


# ============================================================
# 11. Force exact sample-submission column order
# ============================================================

expected_columns = list(
    sample_submission.columns
)

missing_columns = [
    col
    for col in expected_columns
    if col not in prediction_df.columns
]

extra_columns = [
    col
    for col in prediction_df.columns
    if col not in expected_columns
]

if missing_columns:
    raise RuntimeError(
        f"Missing submission columns: "
        f"{missing_columns}"
    )

if extra_columns:
    print(
        "Extra columns removed:",
        extra_columns
    )

submission = prediction_df[
    expected_columns
].copy()


# ============================================================
# 12. Final validation
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "SUBMISSION VALIDATION"
)

print(
    "=" * 70
)

print(
    "Rows:",
    len(submission)
)

print(
    "Expected rows:",
    len(sample_submission)
)

print(
    "Columns:"
)

print(
    submission.columns.tolist()
)

print(
    "\nMissing values:",
    int(
        submission.isna().sum().sum()
    )
)

prediction_columns = [
    col
    for col in expected_columns
    if col != "StudyInstanceUID"
]

print(
    "Minimum prediction:",
    float(
        submission[
            prediction_columns
        ].min().min()
    )
)

print(
    "Maximum prediction:",
    float(
        submission[
            prediction_columns
        ].max().max()
    )
)

assert len(submission) == len(
    sample_submission
)

assert submission.columns.tolist() == (
    expected_columns
)

assert (
    submission.isna().sum().sum()
    == 0
)

assert (
    submission[prediction_columns]
    .applymap(
        lambda x: 0 <= x <= 1
    )
    .all()
    .all()
)

assert (
    submission["StudyInstanceUID"]
    .tolist()
    ==
    sample_submission[
        "StudyInstanceUID"
    ].tolist()
)

# Save
submission_path = (
    OUTPUT_DIR / "submission.csv"
)

submission.to_csv(
    submission_path,
    index=False
)

print(
    "\n"
    + "=" * 70
)

print(
    "SUBMISSION READY"
)

print(
    "=" * 70
)

print(
    "File:",
    submission_path
)

print(
    "\nPreview:"
)

display(
    submission.head()
)

In [ ]:
# FINAL KAGGLE SUBMISSION FILE
# Save directly in /kaggle/working/

submission_path = "/kaggle/working/submission.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("FINAL FILE:", submission_path)
print("EXISTS:", os.path.exists(submission_path))
print("SIZE:", os.path.getsize(submission_path), "bytes")

print("\nSubmission preview:")
display(submission.head())